In [1]:
from src.v1.units import ureg
from src.v5.problem import symbolic, wrap_sympy_function, sp_abs
from src.v6.problem import ConstraintSet, FunctionalSet
from src.v4.torchdata import load_vals, unload_vals, print_formatted_table
from src.v2.tearing import scc
from graph.graphutils import flat_graph_formulation
from graph.operators import sort_scc
from sympy.utilities.lambdify import implemented_function
from scipy import optimize
import sympy as sp
import numpy as np
import torch

In [2]:
def torch_linear_interp1d(x_new, x, y):
    # Assumes x is 1D, sorted, and no duplicate values
    idxs = torch.searchsorted(x, x_new, right=True)
    idxs = torch.clamp(idxs, 1, x.size(0)-1)
    x0, x1 = x[idxs-1], x[idxs]
    y0, y1 = y[idxs-1], y[idxs]
    weight = (x_new - x0) / (x1 - x0)
    return y0 + weight * (y1 - y0)

In [3]:
z_table = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 25, 30, 40])*1e3
T_table_celsius = torch.tensor([15, 8.5,2,-4.49,-10.98,-17.47,-23.96,-30.45,-36.94,-43.42,-49.9,-56.5,-56.5,-51.6,-46.64,-22.8])
T_table = ureg.Quantity(T_table_celsius, 'degC').to('K').magnitude
G_table = torch.tensor([9.807,9.804,9.801,9.797,9.794,9.791,9.788,9.785,9.782,9.779,9.776,9.761,9.745,9.73,9.715,9.684])
P_table = torch.tensor([10.13,8.988,7.95,7.012,6.166,5.405,4.722,4.111,3.565,3.08,2.65,1.211,0.5529,0.2549,0.1197,0.0287])*1e4
rho_table = torch.tensor([1.225,1.112,1.007,0.9093,0.8194,0.7364,0.6601,0.59,0.5258,0.4671,0.4135,0.1948,0.08891,0.04008,0.01841,0.003996])

Tinterp = lambda x: torch_linear_interp1d(x, z_table, T_table)
Ginterp = lambda x: torch_linear_interp1d(x, z_table, G_table)
Pinterp = lambda x: torch_linear_interp1d(x, z_table, P_table)
rhointerp = lambda x: torch_linear_interp1d(x, z_table, rho_table)

In [ ]:
T = wrap_sympy_function(implemented_function(sp.Function('T'), Tinterp))
G = wrap_sympy_function(implemented_function(sp.Function('G'),  Ginterp))
P = wrap_sympy_function(implemented_function(sp.Function('P'), Pinterp))
rho = wrap_sympy_function(implemented_function(sp.Function('rho'),  rhointerp))

In [39]:
z, rhoz, rho_LGz, Wz, Lz, Vz, mrz, mt = symbolic('z', r'\rho_z', r'\rho_{LGz}', 'W_z', 'L_z', 'V_z', r'm_{rz}', 'm_t')
alpha, R, k, mm_He, mm_H2 = 1, 287.05, 1.38064852e-23, 6.64e-27, 1.66e-27 
Apogee = ConstraintSet(
    rhoz == P(z)/(R*T(z)),
    rho_LGz == P(z)*(alpha*mm_He+(1-alpha)*mm_H2)/(k*T(z)),
    Wz == G(z)*mt,
    Lz == Wz,
    Vz == Lz/(G(z)*rhoz),
    mrz == Vz*rho_LGz
)

rz, hz, S, mb = symbolic('r_z', 'h_z', 'S', 'm_b')
p, t_LLDPE, rho_LLDPE = 8/5, 25.4e-6*1, 925 
Materials = ConstraintSet(
    rz == (3*sp_abs(Vz)/(4*np.pi))**(1/3),
    hz == 2*(3/2)*rz,
    S == 4*np.pi*(((rz**2)**p+2*sp_abs(rz)**p*(sp_abs(hz)/2)**p)/3)**(1/p),
    mb == 2*(3/2)*S*t_LLDPE*rho_LLDPE
)

ml, mr0 = symbolic('m_l', 'm_{r0}')
m_vhc, m_parafoil = 4545, 500
Mass = ConstraintSet(
    ml == mr0,
    mt == m_vhc+m_parafoil+ml+mb
)

V0, W0, L0, A0, D, v0 = symbolic('V_0', 'W_0', 'L_0', 'A_0', 'D', 'v_0')
g, rho0, CD = Ginterp(0), rhointerp(0), 0.47
Aero = ConstraintSet(
    W0 == g*mt,
    L0 == g*rho0*V0,
    D == L0-W0,
    A0 == 2*D/(CD*rho0*v0**2)
)

r0, rho_LG0 = symbolic('r_0', r'\rho_{LG0}')
rho_He, rho_H2 = 0.1786, 0.08988 
Geometry = ConstraintSet(
    r0 == 1/np.pi*sp_abs(A0)**0.5,
    V0 == 4/3*np.pi*r0**3,
    rho_LG0 == alpha*rho_He+(1-alpha)*rho_H2,
    mr0 == V0*rho_LG0
)

FPF = FunctionalSet(Apogee, Materials, Mass, Aero, Geometry)

In [40]:
SPF_MDA = FPF.config(parallel=[Apogee, Materials, Mass, Aero, Geometry])
SPF_MDF = FPF.config(elim=[SPF_MDA])
f_MDA = SPF_MDA.build()

In [41]:
x0 = {"z": 10e3, "m_t": 9e3, "v_0": 6, "V_0":1, "m_{r0}":1, "A_0": 2000}
x0val = load_vals(x0, f_MDA.indices, isdict=True, default=1.1)
x0_MDA = f_MDA.analysis(x0val)

In [42]:
fP = Apogee.build()
xP = load_vals(unload_vals(x0_MDA, f_MDA.indices), fP.indices, isdict=True)
fP.analysis(xP)

tensor([8.9445e+02, 5.7087e-02, 1.5668e+04, 1.0000e+04, 6.3339e+04, 4.1352e-01,
        6.3339e+04, 6.4791e+03], dtype=torch.float64)

In [43]:
for elt in SPF_MDF.supersets:
    fP = elt.build()
    xP = load_vals(unload_vals(x0_MDA, f_MDA.indices), fP.indices, isdict=True)
    print_formatted_table([fP.analysis(xP)], fP.indices)

m_{rz}  \rho_{LGz} V_z    z     L_z    \rho_z W_z    m_t     
894.447 0.057      1.57e4 10000 6.33e4 0.414  6.33e4 6479.064
m_b     S        r_z    h_z    V_z   
287.286 4075.846 15.523 46.569 1.57e4
m_t      m_l      m_b     m_{r0}  
6479.064 1146.778 287.286 1146.778
A_0      D      v_0 W_0    L_0    V_0      m_t     
1312.117 1.36e4 6   6.35e4 7.71e4 6420.928 6479.064
m_{r0}   \rho_{LG0} V_0      r_0   A_0     
1146.778 0.179      6420.928 11.53 1312.117


In [14]:
idxrev = {var.item():key for key,var in f_MDF.indices.items()}
in_outs = {elt.supersets[0][0]: elt.build(indices=f_MDF.indices).analysis.structure for elt in SPF_MDF.supersets}
Ein = {eqid: tuple(str(idxrev[idx.item()]) for idx in item[0]) for eqid, item in in_outs.items()}
Eout = {eqid: tuple(str(idxrev[idx.item()]) for idx in item[1]) for eqid, item in in_outs.items()}
edges = Ein, Eout, {}

In [21]:
G = flat_graph_formulation(Ein, Eout)
order = sort_scc(G)

In [ ]:
Loop1 = FunctionalSet(Apogee, Materials, Mass).config(elim=[Apogee, Materials, Mass])
Loop2 = FunctionalSet(Aero, Geometry).config(residuals=[Aero, Geometry])

In [7]:
f_MDF = SPF_MDF.build()
print(f_MDF.analysis.structure), print({f_MDF.idxrev[elt.item()] for elt in f_MDF.analysis.structure[0]})

(tensor([ 0,  1,  5,  9, 19]), tensor([20, 17, 21, 18, 15, 16, 13, 14, 12, 11, 10,  9,  7,  8,  6,  4,  3,  1,
         2,  0]))
{v_0, m_t, V_0, m_{r0}, z}


(None, None)

In [8]:
x0 = {"z": 10e3, "mt": 10e3, "v0": 6, "V0":1, "m_{r0}":1}
x0_MDA = f_MDF.analysis(load_vals(x0, f_MDF.indices, isdict=True, default=1.1))
for elt in SPF_MDF.supersets:
    fP = elt.build()
    xP = load_vals(unload_vals(x0_MDA, f_MDF.indices), fP.indices, isdict=True)
    print_formatted_table([xP], fP.indices)

m_{rz} V_z  \rho_{LGz} L_z    z     \rho_z W_z    m_t     
0      2.66 0          10.754 10000 0.414  10.754 5046.881
m_b   S      r_z  h_z   V_z 
0.881 12.497 0.86 2.579 2.66
m_t      m_l m_b   m_{r0}
5046.881 1   0.881 1.29e6
A_0     v_0 D       W_0    L_0    V_0    m_t     
-1.42e5 1.1 -4.95e4 4.95e4 13.215 7.23e6 5046.881
m_{r0} V_0    \rho_{LG0} r_0     A_0    
1.29e6 7.23e6 0.179      119.971 -1.42e5
